In [1]:
from pathlib import Path
PROJECT_ROOT_OVERRIDE = ""   # Notebook proje ana klasöründe değilse buraya tam yolu gir
B = 1500                   # Makalede bildirilen bootstrap tekrar sayısı
BOOTSTRAP_SEED = 20261121
RUN_BOOTSTRAP = True      # Yalnızca ilk bütünlük denetimi tamamlanınca çalışır
print('V13 — eğitim/inference/GPU kullanılmayacak.')

V13 — eğitim/inference/GPU kullanılmayacak.


In [2]:
import json, hashlib, time
import numpy as np, pandas as pd
from sklearn.metrics import f1_score, accuracy_score, balanced_accuracy_score, confusion_matrix
from IPython.display import display

start = Path(PROJECT_ROOT_OVERRIDE).expanduser() if PROJECT_ROOT_OVERRIDE else Path.cwd()
R = next((p for p in [start,*start.parents] if (p/'outputs'/'asama15c_external').is_dir() and (p/'data'/'external').is_dir()), None)
if R is None: raise FileNotFoundError('Proje kökü bulunamadı. PROJECT_ROOT_OVERRIDE ayarını kontrol et.')
R=R.resolve()
OUT=R/'review_results_external_consistency_v13';OUT.mkdir(parents=True,exist_ok=True)
RET=R/'review_results_ret_found_only_v10'
CNN=R/'review_results_external_BNE_SNE_v6'
OLD=R/'outputs'/'asama15c_external'
SEEDS=(42,1,7)
DATASETS=('PBC','Raabin_Train_TestA')
MODELS=('convnext_tiny','resnet50','retfound_fullft')
NAMES=('basophil','eosinophil','lymphocyte','monocyte','neutrophil')
FULL=('BA','BL','BNE','EO','LY','MMY','MO','MY','PC','PLY','PMY','SNE','VLY')
KEEP=('BA','EO','LY','MO','SNE')
KEEP_IDS=[FULL.index(x) for x in KEEP]
BNE=FULL.index('BNE'); SNE=FULL.index('SNE')
EXPN={'PBC':10298,'Raabin_Train_TestA':14514}
EXPC={'PBC':[1218,3117,1214,1420,3329], 'Raabin_Train_TestA':[301,1066,3461,795,8891]}
EXT={'.jpg','.jpeg','.png','.bmp','.tif','.tiff','.webp'}
print('ROOT:',R,'\nOUTPUT:',OUT)
print('Kaydedilmiş V10:',RET.is_dir(),'| V6:',CNN.is_dir())

ROOT: C:\Users\sarah\Desktop\scı5-wbc 
OUTPUT: C:\Users\sarah\Desktop\scı5-wbc\review_results_external_consistency_v13
Kaydedilmiş V10: True | V6: True


In [3]:
from PIL import Image

PBC_DIR=R/'data/external/PBC_dataset_normal_DIB'
RAABIN_DIR=R/'data/external/Raabin_WBC'

def list_pbc():
    # Aşama15C'deki ham aday taraması; sınıf ve dosya sırası korunur.
    return [(p,y) for y,name in enumerate(NAMES)
            for p in sorted((PBC_DIR/name).rglob('*'))
            if p.is_file() and p.suffix.lower() in EXT and '__MACOSX' not in p.parts]

def list_raabin():
    out=[];ids={name.lower():i for i,name in enumerate(NAMES)}
    for split in ('Train','TestA'):
        root=RAABIN_DIR/split
        for p in sorted(root.rglob('*')):
            if not p.is_file() or p.suffix.lower() not in EXT: continue
            labels=[ids[t.lower()] for t in p.relative_to(root).parts[:-1] if t.lower() in ids]
            if labels:out.append((p,labels[-1]))
    return out

def filter_asama15c_valid_images(candidate_entries, dataset_name):
    """Aşama15C ExternalDataset.__init__ ile AYNI doğrulama/filtreleme.
    Hatalı dosyayı atlar; sıralamayı ve sağlam dosyaların etiketini ASLA değiştirmez.
    """
    valid=[];bad=[]
    for p,y in candidate_entries:
        try:
            with Image.open(p) as im:
                im.verify()
            valid.append((p,int(y)))
        except Exception as exc:
            bad.append({'dataset':dataset_name,'path':str(p),
                        'label_id':int(y),'label_name':NAMES[int(y)],
                        'error_type':type(exc).__name__,'error':str(exc)[:600]})
    return valid,bad

CANDIDATES={'PBC':list_pbc(),'Raabin_Train_TestA':list_raabin()}
ENTRIES={};REJECTED={};manifest_rows=[];bad_rows=[]
for dn in DATASETS:
    valid,bad=filter_asama15c_valid_images(CANDIDATES[dn],dn)
    ENTRIES[dn]=valid;REJECTED[dn]=bad;bad_rows.extend(bad)
    counts=np.bincount(np.asarray([lab for _,lab in valid],dtype='int64'),minlength=5)
    info={'dataset':dn,'candidate_images':len(CANDIDATES[dn]),
          'valid_images':len(valid),'excluded_unreadable_images':len(bad),
          'expected_images':EXPN[dn],'class_counts':str(counts.tolist()),
          'expected_class_counts':str(EXPC[dn])}
    manifest_rows.append(info)
    print(f'{dn}: aday={len(CANDIDATES[dn])}, okunabilir={len(valid)}, '
          f'çıkarılan={len(bad)}, sınıf sayıları={counts.tolist()}')
    for x in bad[:5]:
        print('  Aşama15C ile çıkarılan okunamayan dosya:',x['path'],'->',x['error_type'],x['error'][:100])

pd.DataFrame(bad_rows,columns=['dataset','path','label_id','label_name','error_type','error']).to_csv(
    OUT/'external_unreadable_files_v13.csv',index=False,encoding='utf-8-sig')
pd.DataFrame(manifest_rows).to_csv(OUT/'external_manifest_audit_v13.csv',index=False,encoding='utf-8-sig')

for dn,ent in ENTRIES.items():
    y=np.asarray([lab for _,lab in ent],dtype='int64')
    counts=np.bincount(y,minlength=5).tolist()
    if len(ent)!=EXPN[dn] or counts!=EXPC[dn]:
        raise RuntimeError(f'{dn}: Aşama15C verify filtresinden SONRA bile veri uyumsuz. '
                           f'Aday={len(CANDIDATES[dn])}, geçerli={len(ent)}, '
                           f'dışlanan={len(REJECTED[dn])}, etiketler={counts}; '
                           f'beklenen={EXPN[dn]},{EXPC[dn]}. '
                           f'external_manifest_audit_v13.csv ve external_unreadable_files_v13.csv dosyalarını incele. '
                           'Keyfî dosya seçimi YAPILMADI.')
    print('PASS geçerli manifest:',dn,len(ent),counts)

def manifest_hash(entries):
    h=hashlib.sha256()
    for p,y in entries:
        h.update((str(p.resolve())+'|'+str(y)+'\n').encode('utf8'))
    return h.hexdigest()


PBC: aday=10299, okunabilir=10298, çıkarılan=1, sınıf sayıları=[1218, 3117, 1214, 1420, 3329]
  Aşama15C ile çıkarılan okunamayan dosya: C:\Users\sarah\Desktop\scı5-wbc\data\external\PBC_dataset_normal_DIB\neutrophil\.DS_169665.jpg -> UnidentifiedImageError cannot identify image file 'C:\\Users\\sarah\\Desktop\\scı5-wbc\\data\\external\\PBC_dataset_normal_
Raabin_Train_TestA: aday=14514, okunabilir=14514, çıkarılan=0, sınıf sayıları=[301, 1066, 3461, 795, 8891]
PASS geçerli manifest: PBC 10298 [1218, 3117, 1214, 1420, 3329]
PASS geçerli manifest: Raabin_Train_TestA 14514 [301, 1066, 3461, 795, 8891]


In [4]:
DF_RET=pd.read_csv(RET/'retfound_external_per_seed_v10.csv') if (RET/'retfound_external_per_seed_v10.csv').exists() else pd.DataFrame()
DF_CNN=pd.read_csv(CNN/'external_BNE_SNE_per_seed_v6.csv') if (CNN/'external_BNE_SNE_per_seed_v6.csv').exists() else pd.DataFrame()

PRED={};Q13={}; records=[]; faults=[]

def push_fault(dataset,model,seed,msg):
    faults.append({'dataset':dataset,'model':model,'seed':seed,'error':str(msg)[:600]})

def checked_score(y,p):return float(f1_score(y,p,labels=list(range(5)),average='macro',zero_division=0))

def load_for_run(dn,model,seed):
    d=(RET/dn/f'seed{seed}') if model=='retfound_fullft' else (CNN/dn/model/f'seed{seed}')
    rowset=DF_RET if model=='retfound_fullft' else DF_CNN
    match=rowset[(rowset['dataset']==dn)&(rowset['model']==model)&(rowset['seed']==seed)] if not rowset.empty else pd.DataFrame()
    if len(match)!=1:raise ValueError(f'CSV kaynak satırı sayısı {len(match)} (1 gerekli)')
    source=match.iloc[0]
    def arr(name):
        p=d/name
        if not p.is_file():raise FileNotFoundError(str(p))
        return np.load(p,allow_pickle=False)
    q=arr('probabilities_13class.npy')
    y=arr('true_5class.npy')
    p0=arr('pred_baseline_5class.npy')
    p1=arr('pred_BNEplusSNE_5class.npy')
    n=EXPN[dn]
    if q.shape!=(n,13) or y.shape!=(n,) or p0.shape!=(n,) or p1.shape!=(n,):
        raise ValueError(f'boyut hatası q/y/p0/p1: {q.shape},{y.shape},{p0.shape},{p1.shape}')
    if not np.isfinite(q).all() or q.min()<-1e-7 or q.max()>1.0000001:
        raise ValueError('olasılıklarda NaN/inf/geçersiz değer')
    if np.max(np.abs(q.sum(axis=1)-1))>2e-5:raise ValueError('13-sınıf softmax satır toplamı 1 değil')
    expected_paths=[str(p) for p,_ in ENTRIES[dn]]
    pathfile=d/'paths.txt'
    if not pathfile.is_file():raise FileNotFoundError(str(pathfile))
    actual_paths=pathfile.read_text(encoding='utf8').splitlines()
    if actual_paths!=expected_paths:raise ValueError('paths.txt eski Aşama 15C görüntü manifestiyle uyumsuz')
    expected_y=np.array([lab for _,lab in ENTRIES[dn]],dtype=int)
    if not np.array_equal(y,expected_y):raise ValueError('gerçek etiket/sıralama farklı')
    if list(np.bincount(y,minlength=5))!=EXPC[dn]:raise ValueError('sınıf dağılımları farklı')
    raw=q[:,KEEP_IDS]
    if np.any(raw.sum(1)<=0):raise ValueError('five-class softmax sıfır')
    base=raw/raw.sum(axis=1,keepdims=True)
    merged=np.column_stack([q[:,FULL.index(k)] for k in ('BA','EO','LY','MO')]+[q[:,BNE]+q[:,SNE]])
    merged/=merged.sum(axis=1,keepdims=True)
    if not np.array_equal(base.argmax(axis=1),p0):raise ValueError('baseline NPY ve 13-sınıf olasılıklar uyuşmuyor')
    if not np.array_equal(merged.argmax(axis=1),p1):raise ValueError('BNE+SNE NPY ve 13-sınıf olasılıklar uyuşmuyor')
    f0,f1=checked_score(y,p0),checked_score(y,p1)
    if abs(f0-float(source['baseline_macro_f1']))>1e-6 or abs(f1-float(source['bne_sne_macro_f1']))>1e-6:
        raise ValueError('NPY yeniden hesaplanan Macro-F1 mevcut seed CSV ile uyuşmuyor')
    if abs((f1-f0)-float(source['delta_macro_f1']))>1e-6:
        raise ValueError('delta F1 CSV ile uyuşmuyor')
    changes=int(np.count_nonzero(p0!=p1))
    if changes!=int(source['changed_predictions']):raise ValueError('değişen tahmin sayısı uyuşmuyor')
    recall0=float(np.mean(p0[y==4]==4));recall1=float(np.mean(p1[y==4]==4))
    col0='baseline_neutrophil_recall'
    col1='bne_sne_neutrophil_recall' if model=='retfound_fullft' else 'new_neutrophil_recall'
    if abs(recall0-float(source[col0]))>1e-6 or abs(recall1-float(source[col1]))>1e-6:
        raise ValueError('nötrofil recall CSV ile uyuşmuyor')
    if model!='retfound_fullft':
        cp=Path(str(source['checkpoint']))
        if not cp.is_file() or cp.stat().st_size!=int(source['checkpoint_size']):
            raise ValueError('V6 CNN checkpoint bulunamadı veya kaynak CSV boyutundan farklı')
    if model=='retfound_fullft':
        meta=json.loads((d/'run_metadata.json').read_text(encoding='utf8'))
        if meta.get('manifest_sha256')!=manifest_hash(ENTRIES[dn]):raise ValueError('V10 manifest SHA256 farklı')
        if meta.get('global_pool') is not True or meta.get('forward_method')!='explicit_retinal_mae_encoder_global_pool':
            raise ValueError('V10 forward yöntemi metadata ile uyuşmuyor')
        cp=Path(meta['checkpoint'])
        if not cp.is_file() or cp.stat().st_size!=meta['checkpoint_size'] or cp.stat().st_mtime_ns!=meta['checkpoint_mtime_ns']:
            raise ValueError('checkpoint bulunamadı veya V10 sonrası değişmiş')
        if str(cp)!=str(source['checkpoint']):raise ValueError('checkpoint yolu V10 csv ile farklı')
    # Eski 15C ile etiket/sıra karşılaştırması (eski model tahminleri farklı olabilir)
    oldy=OLD/dn/model/f'seed{seed}'/'true.npy'
    old_labels_match=None
    if oldy.is_file():
        oy=np.load(oldy,allow_pickle=False)
        old_labels_match=bool(np.array_equal(oy,y))
        if not old_labels_match:raise ValueError('Eski Aşama 15C true.npy aynı görüntü sırasına uymuyor')
    return {'dataset':dn,'model':model,'seed':seed,'n':n,'status':'PASS_CACHED_EXTERNAL',
            'baseline_macro_f1':f0,'bne_sne_macro_f1':f1,'delta_macro_f1':f1-f0,
            'changed_predictions':changes,'baseline_neutrophil_recall':recall0,'bne_sne_neutrophil_recall':recall1,
            'old_label_order_matches':old_labels_match,'source_historical_status':source['status'],
            'max_row_sum_error':float(np.max(np.abs(q.sum(1)-1)))}, (y,p0,p1)

for dn in DATASETS:
    for model in MODELS:
        for seed in SEEDS:
            try:
                record,arrays=load_for_run(dn,model,seed)
                records.append(record)
                PRED[dn,model,seed]=arrays
                print('PASS',dn,model,seed)
            except Exception as exc:
                push_fault(dn,model,seed,type(exc).__name__+': '+str(exc))
                print('REVIEW',dn,model,seed, type(exc).__name__,str(exc)[:180])

pd.DataFrame(records).to_csv(OUT/'external_per_seed_verified_v13.csv',index=False,encoding='utf-8-sig')
pd.DataFrame(faults,columns=['dataset','model','seed','error']).to_csv(OUT/'external_audit_issues_v13.csv',index=False,encoding='utf-8-sig')
print('\nDENETİM SONUCU',len(records),'/18 PASS,',len(faults),'REVIEW')
if records:display(pd.DataFrame(records).sort_values(['dataset','model','seed']))
if faults:display(pd.DataFrame(faults))

PASS PBC convnext_tiny 42
PASS PBC convnext_tiny 1
PASS PBC convnext_tiny 7
PASS PBC resnet50 42
PASS PBC resnet50 1
PASS PBC resnet50 7
PASS PBC retfound_fullft 42
PASS PBC retfound_fullft 1
PASS PBC retfound_fullft 7
PASS Raabin_Train_TestA convnext_tiny 42
PASS Raabin_Train_TestA convnext_tiny 1
PASS Raabin_Train_TestA convnext_tiny 7
PASS Raabin_Train_TestA resnet50 42
PASS Raabin_Train_TestA resnet50 1
PASS Raabin_Train_TestA resnet50 7
PASS Raabin_Train_TestA retfound_fullft 42
PASS Raabin_Train_TestA retfound_fullft 1
PASS Raabin_Train_TestA retfound_fullft 7

DENETİM SONUCU 18 /18 PASS, 0 REVIEW


,dataset,model,seed,n,status,baseline_macro_f1,bne_sne_macro_f1,delta_macro_f1,changed_predictions,baseline_neutrophil_recall,bne_sne_neutrophil_recall,old_label_order_matches,source_historical_status,max_row_sum_error
1,PBC,convnext_tiny,1,10298,PASS_CACHED_EXTERNAL,0.967699,0.965926,-0.001773,16,0.999399,0.999399,True,REVIEW_NUMERIC_DRIFT_SAME_PREDICTIONS,3.576279e-07
2,PBC,convnext_tiny,7,10298,PASS_CACHED_EXTERNAL,0.974768,0.972147,-0.002621,26,0.997897,0.998198,True,PASS_SAVED_PREDICTIONS_AND_PROBABILITIES,3.576279e-07
0,PBC,convnext_tiny,42,10298,PASS_CACHED_EXTERNAL,0.952497,0.950207,-0.002290,25,0.995494,0.995795,True,REVIEW_NUMERIC_DRIFT_SAME_PREDICTIONS,2.980232e-07
4,PBC,resnet50,1,10298,PASS_CACHED_EXTERNAL,0.939606,0.937883,-0.001724,18,0.999399,0.999399,True,REVIEW_NUMERIC_DRIFT_SAME_PREDICTIONS,2.980232e-07
5,PBC,resnet50,7,10298,PASS_CACHED_EXTERNAL,0.928689,0.927568,-0.001121,9,0.999099,0.999099,True,REVIEW_NUMERIC_DRIFT_SAME_PREDICTIONS,2.384186e-07
3,PBC,resnet50,42,10298,PASS_CACHED_EXTERNAL,0.908083,0.906988,-0.001095,9,0.998498,0.998498,True,REVIEW_NUMERIC_DRIFT_SAME_PREDICTIONS,2.980232e-07
7,PBC,retfound_fullft,1,10298,PASS_CACHED_EXTERNAL,0.880700,0.879188,-0.001512,15,0.998498,0.998498,True,REVIEW_OLD_CLASSIFICATIONS_CHANGED,3.576279e-07
8,PBC,retfound_fullft,7,10298,PASS_CACHED_EXTERNAL,0.866292,0.866203,-0.000090,2,0.998798,0.998798,True,REVIEW_OLD_CLASSIFICATIONS_CHANGED,2.980232e-07
6,PBC,retfound_fullft,42,10298,PASS_CACHED_EXTERNAL,0.873178,0.873178,0.000000,0,0.996996,0.996996,True,REVIEW_OLD_CLASSIFICATIONS_CHANGED,2.980232e-07
10,Raabin_Train_TestA,convnext_tiny,1,14514,PASS_CACHED_EXTERNAL,0.763541,0.760405,-0.003137,94,0.984704,0.985941,True,REVIEW_NUMERIC_DRIFT_SAME_PREDICTIONS,2.980232e-07


## 3 — Üç seed ortalamaları ve model sıralaması
Yalnızca **18/18** bütünlük kontrolü başarılıysa hesaplanır. Güncellenmiş V10 RETFound skorları seçilir; eski RETFound dış test skorları asla sessizce birleştirilmez.

In [5]:
ALL_PASS=(len(records)==18 and len(faults)==0)
summary=pd.DataFrame()
if ALL_PASS:
    d=pd.DataFrame(records)
    summary=d.groupby(['dataset','model'],as_index=False).agg(
      n_seeds=('seed','nunique'),n_images=('n','first'),
      baseline_mean=('baseline_macro_f1','mean'),baseline_sample_sd=('baseline_macro_f1','std'),
      bne_sne_mean=('bne_sne_macro_f1','mean'),bne_sne_sample_sd=('bne_sne_macro_f1','std'),
      delta_mean=('delta_macro_f1','mean'),delta_sample_sd=('delta_macro_f1','std'),
      neutrophil_recall_baseline_mean=('baseline_neutrophil_recall','mean'),
      neutrophil_recall_bne_sne_mean=('bne_sne_neutrophil_recall','mean'))
    summary['rank_baseline']=summary.groupby('dataset')['baseline_mean'].rank(ascending=False,method='min').astype(int)
    summary.to_csv(OUT/'external_model_summary_verified_v13.csv',index=False,encoding='utf-8-sig')
    display(summary)
else: print('Özet hesaplanmadı: 18/18 PASS gereklidir.')

,dataset,model,n_seeds,n_images,baseline_mean,baseline_sample_sd,bne_sne_mean,bne_sne_sample_sd,delta_mean,delta_sample_sd,neutrophil_recall_baseline_mean,neutrophil_recall_bne_sne_mean,rank_baseline
0,PBC,convnext_tiny,3,10298,0.964988,0.011380,0.962760,0.011307,-0.002228,0.000427,0.997597,0.997797,1
1,PBC,resnet50,3,10298,0.925460,0.016008,0.924146,0.015729,-0.001313,0.000356,0.998999,0.998999,2
2,PBC,retfound_fullft,3,10298,0.873390,0.007206,0.872856,0.006499,-0.000534,0.000848,0.998098,0.998098,3
3,Raabin_Train_TestA,convnext_tiny,3,14514,0.707564,0.074689,0.703636,0.075695,-0.003928,0.001007,0.974956,0.976343,1
4,Raabin_Train_TestA,resnet50,3,14514,0.620858,0.053926,0.613465,0.048524,-0.007393,0.008126,0.978593,0.978967,2
5,Raabin_Train_TestA,retfound_fullft,3,14514,0.440827,0.025000,0.440901,0.025731,0.000074,0.001208,0.934053,0.937690,3


## 4 — 1500 sınıf-katmanlı eşlenik bootstrap (CPU)

Makalenin Methods bölümündeki **model A'nın üç seed ortalama Macro-F1 − model B'nin üç seed ortalama Macro-F1** tanımını kullanır. Her bootstrap tekrarında her iki modelin üç seed'i için **aynı resample indeksleri** uygulanır. Eski Aşama 16'daki konsensüs tahminlerinin farkı ile aynı değildir. Sonuçlar düzeltilecek Table 5 için kullanılabilir; ancak WBCBench film düzeyi bootstrap'ının yerine geçmez.

Tam karşılaştırmalar yalnızca 18/18 cached test PASS koşulunda üretilir.

In [6]:
def macro_from_predictions(y,p):
    conf=np.bincount(5*y.astype('int64')+p.astype('int64'),minlength=25).reshape(5,5)
    tp=np.diag(conf).astype(float)
    den=conf.sum(axis=0)+conf.sum(axis=1)
    return float(np.divide(2*tp,den,out=np.zeros(5,dtype=float),where=den>0).mean())

for yt,pt in [(np.array([0,0,1,2,3,4]),np.array([0,1,1,2,4,4])), (np.array([0,0,0,0,0]),np.array([0,0,0,0,0]))]:
    assert abs(macro_from_predictions(yt,pt)-checked_score(yt,pt))<1e-12
print('Macro-F1 fonksiyonu sentetik test: PASS')

PAIRS=(('convnext_tiny','resnet50'),('convnext_tiny','retfound_fullft'),('resnet50','retfound_fullft'))
ci_rows=[]
if ALL_PASS and RUN_BOOTSTRAP:
    for dn in DATASETS:
        yy=PRED[dn,'convnext_tiny',42][0]
        assert all(np.array_equal(PRED[dn,m,s][0],yy) for m in MODELS for s in SEEDS)
        strata=[np.flatnonzero(yy==k) for k in range(5)]
        pred={m:{s:PRED[dn,m,s][1].astype(np.int64) for s in SEEDS} for m in MODELS}
        def seed_mean_f1(model,idx):
            truth=yy[idx]
            return float(np.mean([macro_from_predictions(truth,pred[model][s][idx]) for s in SEEDS]))
        obs={m:seed_mean_f1(m,np.arange(len(yy))) for m in MODELS}
        rng=np.random.default_rng(BOOTSTRAP_SEED+(0 if dn=='PBC' else 1))
        deltas={pair:[] for pair in PAIRS}
        t0=time.time()
        for j in range(B):
            ix=np.concatenate([rng.choice(g,size=len(g),replace=True) for g in strata])
            mscore={m:seed_mean_f1(m,ix) for m in MODELS}
            for pair in PAIRS:deltas[pair].append(mscore[pair[0]]-mscore[pair[1]])
            if (j+1)%300==0:print(dn,'bootstrap',j+1,'/',B)
        for a,b in PAIRS:
            lo,hi=np.quantile(deltas[a,b],[0.025,0.975])
            ci_rows.append({'dataset':dn,'model_A':a,'model_B':b,'delta_macro_f1':obs[a]-obs[b],
               'ci_low_95':float(lo),'ci_high_95':float(hi),'bootstrap_n':B,
               'resampling':'class_stratified_paired_image','estimator':'mean_over_3_seed_macro_f1_difference'})
        print(dn,'CI hesaplama süresi:',round(time.time()-t0,1),'s')
    ci=pd.DataFrame(ci_rows)
    ci.to_csv(OUT/'external_corrected_seed_average_bootstrap_v13.csv',index=False,encoding='utf-8-sig')
    display(ci)
else: print('Bootstrap atlandı: 18/18 PASS sağlanmadı veya RUN_BOOTSTRAP=False.')

Macro-F1 fonksiyonu sentetik test: PASS
PBC bootstrap 300 / 1500
PBC bootstrap 600 / 1500
PBC bootstrap 900 / 1500
PBC bootstrap 1200 / 1500
PBC bootstrap 1500 / 1500
PBC CI hesaplama süresi: 3.4 s
Raabin_Train_TestA bootstrap 300 / 1500
Raabin_Train_TestA bootstrap 600 / 1500
Raabin_Train_TestA bootstrap 900 / 1500
Raabin_Train_TestA bootstrap 1200 / 1500
Raabin_Train_TestA bootstrap 1500 / 1500
Raabin_Train_TestA CI hesaplama süresi: 4.4 s


,dataset,model_A,model_B,delta_macro_f1,ci_low_95,ci_high_95,bootstrap_n,resampling,estimator
0,PBC,convnext_tiny,resnet50,0.039529,0.035934,0.043260,1500,class_stratified_paired_image,mean_over_3_seed_macro_f1_difference
1,PBC,convnext_tiny,retfound_fullft,0.091598,0.086200,0.097021,1500,class_stratified_paired_image,mean_over_3_seed_macro_f1_difference
2,PBC,resnet50,retfound_fullft,0.052069,0.046435,0.057419,1500,class_stratified_paired_image,mean_over_3_seed_macro_f1_difference
3,Raabin_Train_TestA,convnext_tiny,resnet50,0.086706,0.077014,0.096659,1500,class_stratified_paired_image,mean_over_3_seed_macro_f1_difference
4,Raabin_Train_TestA,convnext_tiny,retfound_fullft,0.266737,0.255424,0.278443,1500,class_stratified_paired_image,mean_over_3_seed_macro_f1_difference
5,Raabin_Train_TestA,resnet50,retfound_fullft,0.180031,0.169124,0.191037,1500,class_stratified_paired_image,mean_over_3_seed_macro_f1_difference


## 5 — Son karar dosyası

`PASS_EXTERNAL_CACHED` yalnızca saklanan **PBC/Raabin** tahminleri ile kaynak CSV'lerin, sınıf sırasının ve checkpoint kimlik bilgilerinin **kendi içinde** uyumlu olduğunu belirtir. `PASS_FULL` WBCBench V12 sonucunu otomatik varsaymaz; yöntem hatasını gizlemez. Makaleye aktarırken eski RETFound skorları ve ilgili confidence interval'lar mutlaka güncellenmelidir.

In [7]:
v12_path=R/'review_results_ret_found_validation_v12'/'retfound_verification_decision_v12.json'
v12_status='NOT_AVAILABLE'
if v12_path.is_file():
    try:
        v12_status=json.loads(v12_path.read_text(encoding='utf8')).get('conclusion','UNKNOWN')
    except Exception as exc:v12_status='INVALID_JSON: '+str(exc)[:100]
full_pass=(v12_status=='PASS_FULL_3_SEEDS__ELIGIBLE_FOR_EXTERNAL_PROTOCOL_AUDIT')
decision={
  'audit':'external_consistency_v13',
  'all_18_cached_runs_verified':bool(ALL_PASS),
  'verified_runs':len(records), 'expected_runs':18, 'issues_count':len(faults),
  'external_ci_recomputed':bool(len(ci_rows)==6),
  'wbcbench_v12_full_verification':v12_status,
  'historical_retfound_15c_mismatch':'EXPECTED_METHOD_MISMATCH__NOT_SILENTLY_ACCEPTED',
  'external_tables_recalculation_ready':bool(ALL_PASS and len(ci_rows)==6),
  'results_ready_to_transcribe_after_v12':bool(ALL_PASS and len(ci_rows)==6 and full_pass),
  'manuscript_and_reviewer_response_finished':False,
  'note':'Ayrıca makaledeki tablo, metin ve yanıtlar güncellenmelidir; geçmiş RETFound skorları kesinlikle karıştırılmamalıdır.',
  'output_directory':str(OUT)}
(OUT/'external_consistency_decision_v13.json').write_text(json.dumps(decision,indent=2,ensure_ascii=False),encoding='utf-8')
print(json.dumps(decision,indent=2,ensure_ascii=False))
print('\nGÖNDERİLECEK DOSYALAR:')
for fname in ('external_per_seed_verified_v13.csv','external_model_summary_verified_v13.csv',
              'external_corrected_seed_average_bootstrap_v13.csv','external_audit_issues_v13.csv',
              'external_consistency_decision_v13.json'):
    path=OUT/fname
    print(('✓ ' if path.exists() else '– '),path)

{
  "audit": "external_consistency_v13",
  "all_18_cached_runs_verified": true,
  "verified_runs": 18,
  "expected_runs": 18,
  "issues_count": 0,
  "external_ci_recomputed": true,
  "wbcbench_v12_full_verification": "REVIEW__FULL_NOT_COMPLETELY_VALIDATED",
  "historical_retfound_15c_mismatch": "EXPECTED_METHOD_MISMATCH__NOT_SILENTLY_ACCEPTED",
  "external_tables_recalculation_ready": true,
  "results_ready_to_transcribe_after_v12": false,
  "manuscript_and_reviewer_response_finished": false,
  "note": "Ayrıca makaledeki tablo, metin ve yanıtlar güncellenmelidir; geçmiş RETFound skorları kesinlikle karıştırılmamalıdır.",
  "output_directory": "C:\\Users\\sarah\\Desktop\\scı5-wbc\\review_results_external_consistency_v13"
}

GÖNDERİLECEK DOSYALAR:
✓  C:\Users\sarah\Desktop\scı5-wbc\review_results_external_consistency_v13\external_per_seed_verified_v13.csv
✓  C:\Users\sarah\Desktop\scı5-wbc\review_results_external_consistency_v13\external_model_summary_verified_v13.csv
✓  C:\Users\sarah\D